# Evaluation

Measures the trained network per task and per language: word error rate
after normalization, character error rate for languages written without
spaces, language identification accuracy, and error buckets from exact to
failed.

In [ ]:
import re
import json
import unicodedata
from pathlib import Path
from fractions import Fraction

import numpy as np
import torch
import soundfile as sf

import import_ipynb
from decoding import transcribe, DecodeOptions, model, tokenizer

In [ ]:
def remove_symbols_keep_diacritics(s):
    return "".join(
        " " if unicodedata.category(c)[0] in "MSP" else c
        for c in unicodedata.normalize("NFKC", s)
    )

def normalize_basic(s):
    s = s.lower()
    s = re.sub(r"[<\[][^>\]]*[>\]]", "", s)
    s = re.sub(r"\(([^)]+?)\)", "", s)
    s = remove_symbols_keep_diacritics(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

print(normalize_basic("Hello, World! (aside) [noise] café"))

In [ ]:
ONES = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11,
    "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
}
TENS = {
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
    "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
}
MULTIPLIERS = {
    "hundred": 100, "thousand": 1000, "million": 10 ** 6,
    "billion": 10 ** 9, "trillion": 10 ** 12,
}

def words_to_number(words):
    total = 0
    current = 0
    any_number = False
    for w in words:
        if w in ONES:
            current += ONES[w]
            any_number = True
        elif w in TENS:
            current += TENS[w]
            any_number = True
        elif w == "hundred" and current:
            current *= 100
        elif w in MULTIPLIERS and w != "hundred":
            total += max(current, 1) * MULTIPLIERS[w]
            current = 0
            any_number = True
        elif w == "and":
            continue
        else:
            return None
    if not any_number:
        return None
    return total + current

def normalize_numbers(s):
    tokens = s.split()
    out = []
    i = 0
    while i < len(tokens):
        j = i
        while j < len(tokens) and tokens[j] in (set(ONES) | set(TENS) | set(MULTIPLIERS) | {"and"}):
            j += 1
        n = words_to_number(tokens[i:j]) if j > i else None
        if n is not None:
            out.append(str(n))
            i = j
        else:
            out.append(tokens[i])
            i += 1
    return " ".join(out)

print(normalize_numbers("twelve thousand three hundred forty five customers"))

In [ ]:
CONTRACTIONS = {
    "won't": "will not", "can't": "can not", "let's": "let us",
    "ain't": "aint", "y'all": "you all", "wanna": "want to",
    "gonna": "going to", "gotta": "got to", "'cause": "because",
}
ABBREVIATIONS = {
    "mr": "mister", "mrs": "missus", "st": "saint", "dr": "doctor",
    "prof": "professor", "capt": "captain", "gov": "governor",
    "ald": "alderman", "gen": "general", "sen": "senator",
    "rep": "representative", "pres": "president", "rev": "reverend",
    "hon": "honorable", "esq": "esquire",
}

def load_spelling_table(path="assets/english.json"):
    with open(path) as f:
        return json.load(f)

SPELLING = load_spelling_table()

def normalize_english(s):
    s = s.lower()
    s = re.sub(r"[<\[][^>\]]*[>\]]", "", s)
    s = re.sub(r"\(([^)]+?)\)", "", s)
    for k, v in CONTRACTIONS.items():
        s = s.replace(k, v)
    s = re.sub(r"\b(" + "|".join(ABBREVIATIONS) + r")\.", lambda m: ABBREVIATIONS[m.group(1)], s)
    s = re.sub(r"n't\b", " not", s)
    s = re.sub(r"'re\b", " are", s)
    s = re.sub(r"'ve\b", " have", s)
    s = re.sub(r"'ll\b", " will", s)
    s = re.sub(r"'m\b", " am", s)
    s = remove_symbols_keep_diacritics(s)
    s = normalize_numbers(s)
    s = " ".join(SPELLING.get(w, w) for w in s.split())
    s = re.sub(r"\s+", " ", s)
    return s.strip()

print(normalize_english("Mr. Smith won't pay the $2 fee — he's twenty two"))

In [ ]:
def edit_distance(ref, hyp):
    m, n = len(ref), len(hyp)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            cur = dp[j]
            if ref[i - 1] == hyp[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[n]

def corpus_wer(pairs, normalizer):
    errors, words = 0, 0
    for ref, hyp in pairs:
        r = normalizer(ref).split()
        h = normalizer(hyp).split()
        errors += edit_distance(r, h)
        words += len(r)
    return errors / max(words, 1)

def corpus_cer(pairs, normalizer):
    errors, chars = 0, 0
    for ref, hyp in pairs:
        r = list(normalizer(ref).replace(" ", ""))
        h = list(normalizer(hyp).replace(" ", ""))
        errors += edit_distance(r, h)
        chars += len(r)
    return errors / max(chars, 1)

print(corpus_wer([("the cat sat", "the cat sat"), ("hello world", "hello word")], normalize_basic))

In [ ]:
def load_wave(path):
    wave, sr = sf.read(path, dtype="float32", always_2d=True)
    wave = wave.mean(axis=1)
    if sr != 16000:
        import scipy.signal as sps
        import math
        g = math.gcd(sr, 16000)
        wave = sps.resample_poly(wave, 16000 // g, sr // g).astype(np.float32)
    return wave

def evaluate_split(shard_dir, language=None, task="transcribe", limit=None, every=1):
    pairs = []
    per_utt = []
    shards = sorted(Path(shard_dir).glob("shard-*.jsonl"))
    count = 0
    for shard in shards:
        npz = np.load(str(shard).replace(".jsonl", ".npz"))
        audio = [npz[k] for k in npz.files]
        with open(shard) as f:
            metas = [json.loads(line) for line in f]
        for i, meta in enumerate(metas):
            count += 1
            if count % every != 0:
                continue
            if limit and len(pairs) >= limit:
                break
            segments = transcribe(model, tokenizer, audio[i],
                                  DecodeOptions(language=language or meta["language"], task=task))
            hyp = " ".join(s.text for s in segments)
            ref = meta["translation"] if task == "translate" and meta.get("translation") else meta["text"]
            pairs.append((ref, hyp))
            per_utt.append({
                "utt_id": meta["utt_id"],
                "ref": ref,
                "hyp": hyp,
                "avg_logprob": min((s.avg_logprob for s in segments), default=None),
            })
        if limit and len(pairs) >= limit:
            break
    return pairs, per_utt

In [ ]:
pairs, per_utt = evaluate_split("data/work/test", limit=500)
wer = corpus_wer(pairs, normalize_english)
print(f"WER {wer * 100:.2f}% over {len(pairs)} utterances")

with open("runs/tiny-multitask-v1/test-eval.jsonl", "w") as f:
    for row in per_utt:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
NO_SPACE_LANGUAGES = {"zh", "ja", "th", "lo", "my", "km"}

def evaluate_language(shard_dir, language, limit=200):
    pairs, _ = evaluate_split(shard_dir, language=language, limit=limit)
    if language in NO_SPACE_LANGUAGES:
        return {"language": language, "metric": "cer", "value": corpus_cer(pairs, normalize_basic), "n": len(pairs)}
    return {"language": language, "metric": "wer", "value": corpus_wer(pairs, normalize_basic), "n": len(pairs)}

results = []
for lang_dir in sorted(Path("data/work/test-by-language").glob("*")):
    if not lang_dir.is_dir():
        continue
    r = evaluate_language(lang_dir, lang_dir.name)
    results.append(r)
    print(r)

with open("runs/tiny-multitask-v1/per-language.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
def error_buckets(per_utt, normalizer):
    buckets = {"exact": 0, "near": 0, "degraded": 0, "failed": 0}
    for row in per_utt:
        r = normalizer(row["ref"]).split()
        h = normalizer(row["hyp"]).split()
        if not r:
            continue
        u_wer = edit_distance(r, h) / len(r)
        if u_wer == 0:
            buckets["exact"] += 1
        elif u_wer <= 0.1:
            buckets["near"] += 1
        elif u_wer <= 0.5:
            buckets["degraded"] += 1
        else:
            buckets["failed"] += 1
    return buckets

print(error_buckets(per_utt, normalize_english))

In [ ]:
def worst_examples(per_utt, normalizer, k=10):
    scored = []
    for row in per_utt:
        r = normalizer(row["ref"]).split()
        h = normalizer(row["hyp"]).split()
        if not r:
            continue
        scored.append((edit_distance(r, h) / len(r), row))
    scored.sort(key=lambda x: -x[0])
    return scored[:k]

for u_wer, row in worst_examples(per_utt, normalize_english):
    print(f"{u_wer:5.2f}  ref: {row['ref'][:80]}")
    print(f"       hyp: {row['hyp'][:80]}")